In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/raw/ipl.csv")

C:\Users\disha\AppData\Local\Temp\ipykernel_23828\2189180487.py:1: DtypeWarning: Columns (28,29,30,31,43,46,47,48,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/ipl.csv")


In [4]:
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

In [5]:
match_info = (
    df.groupby("match_id")
    .first()
    .reset_index()
)

match_info = match_info[
    [
        "match_id",
        "date",
        "season",
        "venue",
        "city",
        "toss_winner",
        "toss_decision",
        "match_won_by",
        "player_of_match"
    ]
]

match_info.head()

,match_id,date,season,venue,city,toss_winner,toss_decision,match_won_by,player_of_match
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,Kolkata Knight Riders,BB McCullum
1,335983,2008-04-19,2007/08,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Chennai Super Kings,bat,Chennai Super Kings,MEK Hussey
2,335984,2008-04-19,2007/08,Feroz Shah Kotla,Delhi,Rajasthan Royals,bat,Delhi Daredevils,MF Maharoof
3,335985,2008-04-20,2007/08,Wankhede Stadium,Mumbai,Mumbai Indians,bat,Royal Challengers Bangalore,MV Boucher
4,335986,2008-04-20,2007/08,Eden Gardens,Kolkata,Deccan Chargers,bat,Kolkata Knight Riders,DJ Hussey


In [6]:
innings_score = (
    df.groupby(["match_id", "innings"])
    .last()
    .reset_index()
)

innings_score = innings_score[
    [
        "match_id",
        "innings",
        "batting_team",
        "bowling_team",
        "team_runs",
        "team_wicket"
    ]
]

innings_score.head()

,match_id,innings,batting_team,bowling_team,team_runs,team_wicket
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,222,3
1,335982,2,Royal Challengers Bangalore,Kolkata Knight Riders,82,10
2,335983,1,Chennai Super Kings,Kings XI Punjab,240,5
3,335983,2,Kings XI Punjab,Chennai Super Kings,207,4
4,335984,1,Rajasthan Royals,Delhi Daredevils,129,8


In [7]:
first = innings_score[
    innings_score["innings"] == 1
].copy()

first = first.rename(
    columns={
        "batting_team":"team1",
        "bowling_team":"team2",
        "team_runs":"first_innings_score",
        "team_wicket":"first_innings_wickets"
    }
)

first.head()

,match_id,innings,team1,team2,first_innings_score,first_innings_wickets
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,222,3
2,335983,1,Chennai Super Kings,Kings XI Punjab,240,5
4,335984,1,Rajasthan Royals,Delhi Daredevils,129,8
6,335985,1,Mumbai Indians,Royal Challengers Bangalore,165,6
8,335986,1,Deccan Chargers,Kolkata Knight Riders,110,10


In [8]:
second = innings_score[
    innings_score["innings"] == 2
].copy()

second = second.rename(
    columns={
        "batting_team":"team2",
        "bowling_team":"team1",
        "team_runs":"second_innings_score",
        "team_wicket":"second_innings_wickets"
    }
)

second.head()

,match_id,innings,team2,team1,second_innings_score,second_innings_wickets
1,335982,2,Royal Challengers Bangalore,Kolkata Knight Riders,82,10
3,335983,2,Kings XI Punjab,Chennai Super Kings,207,4
5,335984,2,Delhi Daredevils,Rajasthan Royals,132,1
7,335985,2,Royal Challengers Bangalore,Mumbai Indians,166,5
9,335986,2,Kolkata Knight Riders,Deccan Chargers,112,5


In [9]:
match_summary = match_info.merge(
    first[
        [
            "match_id",
            "team1",
            "team2",
            "first_innings_score",
            "first_innings_wickets"
        ]
    ],
    on="match_id"
)

match_summary = match_summary.merge(
    second[
        [
            "match_id",
            "second_innings_score",
            "second_innings_wickets"
        ]
    ],
    on="match_id"
)

match_summary.head()

,match_id,date,season,venue,city,toss_winner,toss_decision,match_won_by,player_of_match,team1,team2,first_innings_score,first_innings_wickets,second_innings_score,second_innings_wickets
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,Kolkata Knight Riders,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,222,3,82,10
1,335983,2008-04-19,2007/08,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Chennai Super Kings,bat,Chennai Super Kings,MEK Hussey,Chennai Super Kings,Kings XI Punjab,240,5,207,4
2,335984,2008-04-19,2007/08,Feroz Shah Kotla,Delhi,Rajasthan Royals,bat,Delhi Daredevils,MF Maharoof,Rajasthan Royals,Delhi Daredevils,129,8,132,1
3,335985,2008-04-20,2007/08,Wankhede Stadium,Mumbai,Mumbai Indians,bat,Royal Challengers Bangalore,MV Boucher,Mumbai Indians,Royal Challengers Bangalore,165,6,166,5
4,335986,2008-04-20,2007/08,Eden Gardens,Kolkata,Deccan Chargers,bat,Kolkata Knight Riders,DJ Hussey,Deccan Chargers,Kolkata Knight Riders,110,10,112,5


In [10]:
team_mapping = {
    "Delhi Daredevils":"Delhi Capitals",
    "Kings XI Punjab":"Punjab Kings",
    "Rising Pune Supergiant":"Rising Pune Supergiants",
    "Royal Challengers Bangalore":"Royal Challengers Bangaluru",
    "Gujurat Lions":"Gujarat Titans"
}

for col in ["team1","team2","match_won_by","toss_winner"]:
    match_summary[col] = match_summary[col].replace(team_mapping)

match_summary.head()

,match_id,date,season,venue,city,toss_winner,toss_decision,match_won_by,player_of_match,team1,team2,first_innings_score,first_innings_wickets,second_innings_score,second_innings_wickets
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangaluru,field,Kolkata Knight Riders,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangaluru,222,3,82,10
1,335983,2008-04-19,2007/08,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Chennai Super Kings,bat,Chennai Super Kings,MEK Hussey,Chennai Super Kings,Punjab Kings,240,5,207,4
2,335984,2008-04-19,2007/08,Feroz Shah Kotla,Delhi,Rajasthan Royals,bat,Delhi Capitals,MF Maharoof,Rajasthan Royals,Delhi Capitals,129,8,132,1
3,335985,2008-04-20,2007/08,Wankhede Stadium,Mumbai,Mumbai Indians,bat,Royal Challengers Bangaluru,MV Boucher,Mumbai Indians,Royal Challengers Bangaluru,165,6,166,5
4,335986,2008-04-20,2007/08,Eden Gardens,Kolkata,Deccan Chargers,bat,Kolkata Knight Riders,DJ Hussey,Deccan Chargers,Kolkata Knight Riders,110,10,112,5


In [11]:
match_summary["team1_win"] = (
    match_summary["match_won_by"] ==
    match_summary["team1"]
).astype(int)

match_summary["bat_first_won"] = (
    match_summary["match_won_by"] ==
    match_summary["team1"]
).astype(int)

match_summary["chasing_team_won"] = (
    match_summary["match_won_by"] ==
    match_summary["team2"]
).astype(int)

match_summary.head()

,match_id,date,season,venue,city,toss_winner,toss_decision,match_won_by,player_of_match,team1,team2,first_innings_score,first_innings_wickets,second_innings_score,second_innings_wickets,team1_win,bat_first_won,chasing_team_won
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangaluru,field,Kolkata Knight Riders,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangaluru,222,3,82,10,1,1,0
1,335983,2008-04-19,2007/08,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Chennai Super Kings,bat,Chennai Super Kings,MEK Hussey,Chennai Super Kings,Punjab Kings,240,5,207,4,1,1,0
2,335984,2008-04-19,2007/08,Feroz Shah Kotla,Delhi,Rajasthan Royals,bat,Delhi Capitals,MF Maharoof,Rajasthan Royals,Delhi Capitals,129,8,132,1,0,0,1
3,335985,2008-04-20,2007/08,Wankhede Stadium,Mumbai,Mumbai Indians,bat,Royal Challengers Bangaluru,MV Boucher,Mumbai Indians,Royal Challengers Bangaluru,165,6,166,5,0,0,1
4,335986,2008-04-20,2007/08,Eden Gardens,Kolkata,Deccan Chargers,bat,Kolkata Knight Riders,DJ Hussey,Deccan Chargers,Kolkata Knight Riders,110,10,112,5,0,0,1


In [12]:
print(match_summary.columns)

Index(['match_id', 'date', 'season', 'venue', 'city', 'toss_winner',
       'toss_decision', 'match_won_by', 'player_of_match', 'team1', 'team2',
       'first_innings_score', 'first_innings_wickets', 'second_innings_score',
       'second_innings_wickets', 'team1_win', 'bat_first_won',
       'chasing_team_won'],
      dtype='object')


In [13]:
venue_mapping = {

    # Delhi
    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium",
    "Feroz Shah Kotla": "Arun Jaitley Stadium",

    # Bengaluru
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",

    # Mumbai
    "Brabourne Stadium, Mumbai": "Brabourne Stadium",
    "Dr DY Patil Sports Academy, Mumbai": "Dr DY Patil Sports Academy",
    "Wankhede Stadium, Mumbai": "Wankhede Stadium",

    # Mohali / Chandigarh
    "Punjab Cricket Association IS Bindra Stadium": "Punjab Cricket Association Stadium",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Punjab Cricket Association Stadium",
    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association Stadium",
    "Punjab Cricket Association Stadium": "Punjab Cricket Association Stadium",

    # Hyderabad
    "Rajiv Gandhi International Stadium": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium, Hyderabad",

    # Chennai
    "MA Chidambaram Stadium": "MA Chidambaram Stadium, Chennai",
    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium, Chennai",

    # Kolkata
    "Eden Gardens, Kolkata": "Eden Gardens",

    # Jaipur
    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",

    # Ahmedabad
    "Narendra Modi Stadium, Ahmedabad": "Narendra Modi Stadium",
    "Sardar Patel Stadium, Motera": "Narendra Modi Stadium",

    # Lucknow
    "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium": "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow",

    # Guwahati
    "Barsapara Cricket Stadium": "Barsapara Cricket Stadium, Guwahati",

    # Dharamsala
    "Himachal Pradesh Cricket Association Stadium": "Himachal Pradesh Cricket Association Stadium, Dharamsala",

    # Visakhapatnam
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium": "Dr YS Rajasekhara Reddy ACA-VDCA Cricket Stadium",

    # Pune
    "Maharashtra Cricket Association Stadium": "Maharashtra Cricket Association Stadium, Pune",

    # Dubai
    "Dubai International Cricket Stadium, Dubai": "Dubai International Cricket Stadium",

    # Abu Dhabi
    "Sheikh Zayed Stadium, Abu Dhabi": "Sheikh Zayed Stadium",

    # Sharjah
    "Sharjah Cricket Stadium, Sharjah": "Sharjah Cricket Stadium",

    # Centurion
    "SuperSport Park": "SuperSport Park, Centurion",

    # Durban
    "Kingsmead, Durban": "Kingsmead",

    # Port Elizabeth
    "St George's Park": "St George's Park, Port Elizabeth",

    # Kimberley
    "De Beers Diamond Oval, Kimberley": "De Beers Diamond Oval",

    # Bloemfontein
    "OUTsurance Oval": "OUTsurance Oval, Bloemfontein",

    # East London
    "Buffalo Park, East London": "Buffalo Park",

    # Johannesburg
    "New Wanderers Stadium": "The Wanderers Stadium",

    # Cape Town
    "Newlands, Cape Town": "Newlands",

    # Paarl
    "Boland Park, Paarl": "Boland Park",

    # Cuttack
    "Barabati Stadium, Cuttack": "Barabati Stadium",

    # Indore
    "Holkar Cricket Stadium, Indore": "Holkar Cricket Stadium",

    # Kanpur
    "Green Park, Kanpur": "Green Park",

    # Raipur
    "Shaheed Veer Narayan Singh International Stadium": "Shaheed Veer Narayan Singh International Stadium, Raipur",

    # Ranchi
    "JSCA International Stadium Complex": "JSCA International Stadium Complex, Ranchi",

    # Nagpur
    "Vidarbha Cricket Association Stadium, Jamtha": "Vidarbha Cricket Association Stadium",

    # Kochi
    "Nehru Stadium": "Jawaharlal Nehru Stadium, Kochi",

    # Thiruvananthapuram
    "Greenfield International Stadium": "Greenfield International Stadium, Thiruvananthapuram",

    # Vizag alternate
    "ACA-VDCA Stadium": "Dr YS Rajasekhara Reddy ACA-VDCA Cricket Stadium"
}

In [14]:
match_summary["venue"] = match_summary["venue"].replace(venue_mapping)

In [15]:
match_summary.to_csv(
    "../data/processed/match_summary.csv",
    index=False
)

In [16]:
match_summary.shape

(1187, 18)

In [17]:
teams = sorted(
    list(
        set(match_summary["team1"]).union(
            set(match_summary["team2"])
        )
    )
)

print(teams)

['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bangaluru', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


In [18]:
print(match_summary.columns.tolist())

['match_id', 'date', 'season', 'venue', 'city', 'toss_winner', 'toss_decision', 'match_won_by', 'player_of_match', 'team1', 'team2', 'first_innings_score', 'first_innings_wickets', 'second_innings_score', 'second_innings_wickets', 'team1_win', 'bat_first_won', 'chasing_team_won']
